In [7]:
from google.colab import drive
drive.mount('/content/drive')

# Install ALL required dependencies (IMPORTANT)
!pip install deepface tensorflow opencv-python-headless scipy -q

import os, json

# Create base folder if not exists
BASE = '/content/drive/MyDrive/DatingApp_ML'
os.makedirs(BASE, exist_ok=True)

print("✅ Setup Ready!")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 52.2 MB/s eta 0:00:00
✅ Setup Ready!


In [8]:
from google.colab import files
from deepface import DeepFace
import numpy as np
import json

def get_face_embedding(image_path):
    """
    Convert a face photo into a 512-number vector (FaceNet512 model).
    This vector captures the unique features of the face.
    """
    embedding_data = DeepFace.represent(
        img_path=image_path,
        model_name="Facenet512",
        enforce_detection=False
    )
    return embedding_data[0]["embedding"]  # list of 512 floats

# Upload a face photo
print("Upload face photo for User 1:")
f1 = files.upload()
img1_path = list(f1.keys())[0]

# Get embedding
embedding1 = get_face_embedding(img1_path)
print(f"Embedding generated! Shape: {len(embedding1)} dimensions")

# Save embedding (in real app this goes to database per user)
user1_data = {
    "user_id": "user_001",
    "name": "User 1",
    "embedding": embedding1
}
save_path = f'{BASE}/data/user1_embedding.json'
with open(save_path, 'w') as f:
    json.dump(user1_data, f)
print(f"Embedding saved to: {save_path}")


26-04-24 13:40:47 - Directory /root/.deepface has been created
26-04-24 13:40:47 - Directory /root/.deepface/weights has been created
Upload face photo for User 1:


Saving IMG_20250322_160024064.jpg to IMG_20250322_160024064.jpg


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facenet512_weights.h5
To: /root/.deepface/weights/facenet512_weights.h5


26-04-24 13:41:43 - 🔗 facenet512_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facenet512_weights.h5 to /root/.deepface/weights/facenet512_weights.h5...


100%|██████████| 95.0M/95.0M [00:00<00:00, 288MB/s]


Embedding generated! Shape: 512 dimensions
Embedding saved to: /content/drive/MyDrive/DatingApp_ML/data/user1_embedding.json


In [ ]:
from scipy.spatial.distance import cosine

def face_match_percentage(embedding1, embedding2):
    """
    Compare two face embeddings using cosine similarity.
    Returns a percentage (0-100). Higher = more similar.
    """
    # Cosine distance: 0 = identical, 1 = completely different
    distance = cosine(embedding1, embedding2)
    # Convert to percentage similarity
    similarity = round((1 - distance) * 100, 2)
    return max(0, min(100, similarity))
# Upload second face
print("Upload face photo for User 2:")
f2 = files.upload()
img2_path = list(f2.keys())[0]

embedding2 = get_face_embedding(img2_path)

# Calculate match
match_pct = face_match_percentage(embedding1, embedding2)
print(f"\n--- Face Match Result ---")
print(f"Match Percentage: {match_pct}%")

if match_pct > 75:
    print("Very high similarity!")
elif match_pct > 50:
    print("Moderate similarity")
else:
    print("Low similarity")


Upload face photo for User 2:


In [ ]:
import os
import glob

def search_similar_faces(query_embedding, embeddings_dir, top_k=5):
    """
    Given a face embedding, find the most similar faces from all stored users.
    """
    results = []
    all_files = glob.glob(f'{embeddings_dir}/*_embedding.json')

    for filepath in all_files:
        with open(filepath, 'r') as f:
            user_data = json.load(f)

        stored_emb = user_data['embedding']
        similarity = face_match_percentage(query_embedding, stored_emb)

        results.append({
            "user_id"    : user_data['user_id'],
            "name"       : user_data['name'],
            "similarity" : similarity,
            "file"       : filepath
        })

    # Sort by highest similarity
    results.sort(key=lambda x: x['similarity'], reverse=True)
    return results[:top_k]

# Save user2 embedding too
user2_data = {"user_id": "user_002", "name": "User 2", "embedding": embedding2}
with open(f'{BASE}/data/user2_embedding.json', 'w') as f:
  json.dump(user2_data, f)

# Search
matches = search_similar_faces(embedding1, f'{BASE}/data')
print("\n--- Top Matches ---")
for i, m in enumerate(matches, 1):
    print(f"{i}. {m['name']} — {m['similarity']}% match")


--- Top Matches ---
1. User 1 — 100% match
2. User 2 — 35.12% match
